# BODAQS Batch Pre-processing Pipeline

Workflow:
1. Load CSVs
2. Canonicalise signal names
3. Normalize (zero + scale)
4. Estimate velocity/acceleration
5. Load event schema + detect events
6. Calculate event metrics
7. Write artifacts


In [1]:
import pandas as pd
import numpy as np
import logging
import importlib

from pathlib import Path

from bodaqs_analysis import run_macro
from bodaqs_analysis import normalize_and_scale
from bodaqs_analysis import estimate_va
from bodaqs_analysis import load_event_schema
from bodaqs_analysis import detect_events_from_schema
from bodaqs_analysis import extract_metrics_df


import bodaqs_analysis.widgets.event_browser as eb
importlib.reload(eb)

from bodaqs_analysis.artifacts import (
    ArtifactStore,
    make_run_id,
    save_session_artifacts,
    write_run_manifest,
    write_session_manifest,
    write_events_partitioned_by_schema_id,
    write_metrics_partitioned_by_schema_id,
    copy_raw_csv_to_source,
    ensure_run_is_new,
    ensure_session_is_new,
    update_manifest_description,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s"
)

# logging.getLogger("bodaqs_analysis").setLevel(logging.DEBUG)
logging.getLogger("bodaqs_analysis.detect").setLevel(logging.INFO)

logger = logging.getLogger(__name__)


In [2]:
# ---- Inputs ----
CSV_PATH = r"C:\Users\Ben Connor\OneDrive\Documents\logs\2026-02-01_09-48*.CSV"   # change to your log
SCHEMA_PATH = r"event schema\event_schema.yaml"
INGESTION_MODE = "tolerant"
ZEROING_ENABLED = True
PROMPT_FOR_DESCRIPTIONS = True

# Columns and full-ranges (mm) for normalization (edit as needed)
NORMALIZE_RANGES = {
    "front_shock_dom_suspension [mm]": 170.0,
    "rear_shock_dom_suspension [mm]": 165.0,
}

SAMPLE_RATE_HZ = 200   # if known; used for VA estimation when time col is absent


In [3]:
# ---- Batch orchestration + artifact production ----
pattern = Path(CSV_PATH)
CSV_FILES = sorted(pattern.parent.glob(pattern.name))

if not CSV_FILES:
    raise FileNotFoundError(f"No CSV files matched: {CSV_PATH}")

logger.info(f"Found {len(CSV_FILES)} files:")
for p in CSV_FILES:
    logger.info("  %s", p.name)

# ---- Artifact init (one run_id per batch) ----
store = ArtifactStore(Path("artifacts"))
run_id = make_run_id(tz_label="AWST") 
ensure_run_is_new(store, run_id=run_id, force=False)
logger.info("Artifact run_id = %s", run_id)

events_all = []
metrics_all = []
sessions_by_id = {}
schema = None
session_ids_in_run = []

for p in CSV_FILES:
    logger.info("Processing %s ...", p.name)

    results = run_macro(
        str(p),
        SCHEMA_PATH,
        zeroing_enabled=ZEROING_ENABLED,
        normalize_ranges=NORMALIZE_RANGES,
        sample_rate_hz=SAMPLE_RATE_HZ,
        strict=(INGESTION_MODE == "strict"),
    )

    session = results["session"]
    sid = str(session["session_id"])
    ensure_session_is_new(store, run_id=run_id, session_id=sid, force=False)
    sessions_by_id[sid] = session
    session_ids_in_run.append(sid)

    # Copy raw CSV into artifacts
    source_sha256 = copy_raw_csv_to_source(store=store, run_id=run_id, session_id=sid, csv_path=p)

    # Save canonical session artifacts
    save_session_artifacts(
        store,
        run_id=run_id,
        session_id=sid,
        session_df=session["df"],
        session_meta=session["meta"],
    )

    # Write per-session manifest (minimal but useful)
    write_session_manifest(
        store,
        run_id=run_id,
        session_id=sid,
        contracts={"session": "v0.x", "events": "v0.x", "metrics": "v0.x"},
        source={
            "path": "source/input.csv",
            "sha256": source_sha256,
        },
        summary={"n_rows": int(len(session["df"]))},
    )

    # Freeze schema once per run (optional)
    if schema is None:
        schema = results.get("schema", None)

    # Events / Metrics tables (per schema_id)
    ev = results.get("events")
    mt = results.get("metrics")

    if isinstance(ev, pd.DataFrame) and not ev.empty and (ev["session_id"] != sid).any():
        raise ValueError(f"events_df session_id mismatch for sid={sid}")

    if isinstance(mt, pd.DataFrame) and not mt.empty and (mt["session_id"] != sid).any():
        raise ValueError(f"metrics_df session_id mismatch for sid={sid}")

    if isinstance(ev, pd.DataFrame) and not ev.empty:
        events_all.append(ev)
        write_events_partitioned_by_schema_id(
            store=store,
            run_id=run_id,
            session_id=sid,
            events_df=ev,
            schema_path=SCHEMA_PATH,
        )
    
    if isinstance(mt, pd.DataFrame) and not mt.empty:
        metrics_all.append(mt)
        write_metrics_partitioned_by_schema_id(
            store=store,
            run_id=run_id,
            session_id=sid,
            metrics_df=mt,
        )
    

# Batch-level combined dfs (your existing behavior)
events_df  = pd.concat(events_all, ignore_index=True) if events_all else pd.DataFrame()
metrics_df = pd.concat(metrics_all, ignore_index=True) if metrics_all else pd.DataFrame()

logging.debug("events_df: %s", events_df.shape)
logging.debug("metrics_df: %s", metrics_df.shape)

# Write run manifest (once per batch)
write_run_manifest(
    store,
    run_id=run_id,
    session_ids=session_ids_in_run,
    timezone_label="AWST",
    pipeline_config={
        "schema_path": str(SCHEMA_PATH),
        "zeroing_enabled": bool(ZEROING_ENABLED),
        "normalize_ranges": bool(NORMALIZE_RANGES),
        "sample_rate_hz": float(SAMPLE_RATE_HZ),
        "ingestion_mode": str(INGESTION_MODE),
        "n_files": int(len(CSV_FILES)),
    },
)

logger.info("Wrote artifacts for run_id=%s with %d sessions", run_id, len(session_ids_in_run))

if PROMPT_FOR_DESCRIPTIONS:
    from bodaqs_analysis.artifacts import set_run_description, set_session_description

    run_desc = input(f"Run description for {run_id} (blank to skip): ").strip()
    set_run_description(store, run_id=run_id, description=run_desc)

    for sid in session_ids_in_run:
        s_desc = input(f"Session {sid} description (blank to skip, '.' to stop): ").strip()
        if s_desc == ".":
            break
        set_session_description(store, run_id=run_id, session_id=sid, description=s_desc)

INFO:__main__:Found 1 files:
INFO:__main__:  2026-02-01_09-48-24.CSV
INFO:__main__:Artifact run_id = run_2026-02-09T15-05-41_AWST
INFO:__main__:Processing 2026-02-01_09-48-24.CSV ...
INFO:bodaqs_analysis.pipeline:Session load complete: C:\Users\Ben Connor\OneDrive\Documents\logs\2026-02-01_09-48-24.CSV
INFO:bodaqs_analysis.pipeline:Session pre-process complete
INFO:bodaqs_analysis.pipeline:Schema load complete
INFO:bodaqs_analysis.detect:Built EVENTS_DF with 132 rows from 9 schema event(s) → 18 sensor-expanded event(s)
INFO:bodaqs_analysis.pipeline:Event detection complete
INFO:bodaqs_analysis.pipeline:events rows: 132
INFO:bodaqs_analysis.pipeline:Schema events with zero detections this run: ['bottom_out']
INFO:bodaqs_analysis.pipeline:Running segment extraction for detected schema events: ['compressions_25_50', 'compressions_50_75', 'compressions_75_100', 'rebounds_25_50', 'rebounds_50_75', 'rebounds_75_100', 'top_out', 'top_out2']
INFO:bodaqs_analysis.pipeline:Segment extraction com

Run description for run_2026-02-09T15-05-41_AWST (blank to skip):  
Session 2026-02-01_09-48-24 description (blank to skip, '.' to stop):  
